# 🌌 VOICE STUDIO DA GALAXIA

Estúdio de voz em Gradio com o motor **oficial [OmniVoice](https://github.com/k2-fsa/OmniVoice)**
(k2-fsa): clonagem de voz, Voice Design, Auto Voice, 600+ idiomas e narração longa nativa.

**Como usar (Colab):**
1. `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4` (ou outra GPU NVIDIA).
2. Ajuste, se quiser, a célula **0. Configurações**.
3. `Ambiente de execução → Executar tudo` (ou execute as células 0 → 6 em ordem).
4. Abra o link impresso pela célula **6** (proxy do Colab ou link público `gradio.live`).

**Persistência:** pesos do modelo, vozes preparadas (cache), WAVs gerados e as últimas configurações da
interface ficam no Google Drive, em `MyDrive/VOICE_STUDIO_DA_GALAXIA/`. Numa nova sessão basta
executar tudo de novo: nada é baixado outra vez e as vozes continuam em cache.
O que **não** sobrevive ao fim da sessão: os pacotes instalados, os processos (servidor Gradio),
a memória da GPU e o link — eles são recriados automaticamente ao executar as células.

> Use apenas a sua própria voz ou vozes com autorização expressa do titular.

In [ ]:
#@title ⚙️ 0. Configurações (edite e execute)
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "VOICE_STUDIO_DA_GALAXIA"  #@param {type:"string"}
MODEL_ID = "k2-fsa/OmniVoice"  #@param {type:"string"}
#@markdown `pinned` = commit oficial validado · `latest` = branch master atual · `pypi` = última versão publicada (sem FlashInfer)
OMNIVOICE_SOURCE = "pinned"  #@param ["pinned", "latest", "pypi"]
#@markdown FlashInfer: `auto` liga só em GPUs sm_80+ (A100/L4/H100) após autoteste; na T4 mantém o baseline FP16.
FLASHINFER = "auto"  #@param ["auto", "on", "off"]
FLASHINFER_CUDA_GRAPHS = False  #@param {type:"boolean"}
INSTALL_TEXT_NORMALIZATION = True  #@param {type:"boolean"}
PUBLIC_SHARE_LINK = True  #@param {type:"boolean"}
RESTORE_LAST_SETTINGS = True  #@param {type:"boolean"}
PORT = 7860  #@param {type:"integer"}

import sys
IN_COLAB = "google.colab" in sys.modules
APP_DIR = "/content/voice_studio_app" if IN_COLAB else __import__("os").path.abspath("voice_studio_app")
print("Configurações carregadas. Colab:", IN_COLAB)

In [ ]:
#@title 🖥️ 1. Verificar GPU e ambiente
import platform, shutil, subprocess

print("Python", sys.version.split()[0], "·", platform.platform())
if shutil.which("nvidia-smi"):
    print(subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap", "--format=csv"],
        capture_output=True, text=True).stdout)

import torch
print("torch", torch.__version__, "· CUDA", torch.version.cuda, "· GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    free, total = torch.cuda.mem_get_info(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} · {total/1024**3:.1f} GB · sm_{cc[0]}{cc[1]}")
    if cc < (8, 0) and FLASHINFER == "auto":
        print("→ FlashInfer ficará desligado (modo auto); será usado o baseline FP16 seguro para T4.")
else:
    print("⚠️ Nenhuma GPU NVIDIA encontrada. No Colab: Ambiente de execução → Alterar o tipo de "
          "ambiente de execução → GPU T4. Em CPU o estúdio funciona, mas é muito lento.")

In [ ]:
#@title 💾 2. Montar o Google Drive (persistência)
import os

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    STUDIO_HOME = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
else:
    STUDIO_HOME = "/content/voice_studio_data" if IN_COLAB else os.path.abspath("voice_studio_data")
    if USE_GOOGLE_DRIVE:
        print("(fora do Colab: usando pasta local)")

for sub in ("models", "hf_cache", "outputs", "voices"):
    os.makedirs(os.path.join(STUDIO_HOME, sub), exist_ok=True)
# Tudo que o Hugging Face baixar (ex.: Whisper para transcrição automática) também fica no Drive.
os.environ["HF_HOME"] = os.path.join(STUDIO_HOME, "hf_cache")
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

n_voices = len([f for f in os.listdir(os.path.join(STUDIO_HOME, "voices")) if f.endswith(".pt")])
n_wavs = len([f for f in os.listdir(os.path.join(STUDIO_HOME, "outputs")) if f.endswith(".wav")])
print("Pasta do estúdio:", STUDIO_HOME)
print(f"Vozes em cache: {n_voices} · WAVs gerados: {n_wavs} · "
      f"configurações salvas: {os.path.isfile(os.path.join(STUDIO_HOME, 'settings.json'))}")

In [ ]:
#@title 📦 3. Instalar dependências compatíveis
import importlib, importlib.metadata as md, os, subprocess, sys

OMNIVOICE_COMMIT = "08be0b4ccbac3e13e374e86fbfead4b4cac343e2"
GRADIO_VERSION = "5.49.1"
FLASHINFER_VERSION = "0.6.15.post1"

def pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print("$ pip install", " ".join(args))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("pip falhou: " + " ".join(args))
    return r.returncode == 0

def installed(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

omnivoice_spec = {
    "pinned": f"git+https://github.com/k2-fsa/OmniVoice.git@{OMNIVOICE_COMMIT}",
    "latest": "git+https://github.com/k2-fsa/OmniVoice.git",
    "pypi": "omnivoice",
}[OMNIVOICE_SOURCE]
marker = os.path.join(APP_DIR, f".installed-{OMNIVOICE_SOURCE}-{OMNIVOICE_COMMIT[:8]}-{GRADIO_VERSION}")

if os.path.exists(marker):
    print("Dependências já instaladas nesta sessão ✔")
else:
    # Never let pip replace the CUDA build of torch that Colab ships.
    # (pin the *distribution* version: it may lack the "+cuXXX" suffix of torch.__version__)
    torch_ver = installed("torch")
    cu_tag = "cu" + torch.version.cuda.replace(".", "") if torch.version.cuda else None
    constraints = [f"torch=={torch_ver}"]
    ta_ver = installed("torchaudio")
    if ta_ver is None or ta_ver.split("+")[0] != torch_ver.split("+")[0]:
        index = [f"--index-url=https://download.pytorch.org/whl/{cu_tag}"] if cu_tag else []
        pip(f"torchaudio=={torch_ver.split('+')[0]}", *index)
        ta_ver = installed("torchaudio")
    constraints.append(f"torchaudio=={ta_ver}")
    os.makedirs(APP_DIR, exist_ok=True)
    cfile = os.path.join(APP_DIR, "constraints.txt")
    with open(cfile, "w") as f:
        f.write("\n".join(constraints) + "\n")

    pip(omnivoice_spec, f"gradio=={GRADIO_VERSION}", "-c", cfile)
    if INSTALL_TEXT_NORMALIZATION:
        if not pip("WeTextProcessing", "num2words", "-c", cfile, check=False):
            print("⚠️ Normalização de texto indisponível (o checkbox será ignorado com aviso).")

    want_fi = FLASHINFER == "on" or (FLASHINFER == "auto" and torch.cuda.is_available()
                                     and torch.cuda.get_device_capability(0) >= (8, 0))
    if want_fi and OMNIVOICE_SOURCE != "pypi" and cu_tag:
        ok = pip(f"flashinfer-python=={FLASHINFER_VERSION}",
                 f"flashinfer-jit-cache=={FLASHINFER_VERSION}+{cu_tag}",
                 f"--extra-index-url=https://flashinfer.ai/whl/{cu_tag}/", "-c", cfile, check=False)
        print("FlashInfer instalado ✔" if ok else "⚠️ FlashInfer não instalado — será usado o baseline FP16.")
    open(marker, "w").close()

importlib.invalidate_caches()
os.makedirs(os.path.join(APP_DIR, "voice_studio"), exist_ok=True)
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
for name in ("torch", "torchaudio", "transformers", "gradio", "omnivoice", "flashinfer-python", "WeTextProcessing"):
    print(f"  {name:18s} {installed(name) or '—'}")

### 3.b Código do estúdio
As células abaixo gravam o código-fonte da interface e do motor em `APP_DIR/voice_studio/`
(gerado a partir do repositório por `tools/build_notebook.py`). Basta executá-las.

In [ ]:
%%writefile {APP_DIR}/voice_studio/__init__.py
"""VOICE STUDIO DA GALAXIA — Gradio studio for the official OmniVoice TTS engine."""

from .engine import DEFAULTS, StudioEngine, StudioError

__all__ = ["DEFAULTS", "StudioEngine", "StudioError"]
__version__ = "1.0.0"

In [ ]:
%%writefile {APP_DIR}/voice_studio/engine.py
"""Inference engine for VOICE STUDIO DA GALAXIA.

Thin, faithful wrapper around the official ``k2-fsa/OmniVoice`` Python API:

* ``OmniVoice.from_pretrained`` loads the real weights (FP16 on CUDA).
* ``OmniVoice.create_voice_clone_prompt`` encodes the reference voice once;
  the resulting ``VoiceClonePrompt`` is cached in memory and on disk
  (Google Drive in Colab) so repeated generations skip that work.
* ``OmniVoice.generate`` does all synthesis, including the engine's native
  long-form chunking (``audio_chunk_duration`` / ``audio_chunk_threshold``).
* FlashInfer (``omnivoice.models.omnivoice_flashinfer.apply_flashinfer``) is
  enabled only when the GPU supports it and a self-test passes; otherwise the
  plain FP16 path is kept (the "T4-safe baseline").

Nothing here re-implements OmniVoice's decoding: the engine only counts calls
to the model's own iterative decoder to report progress and statistics.
"""

import gc
import hashlib
import json
import logging
import os
import re
import threading
import time
import uuid
from dataclasses import dataclass, field
from typing import Callable, Optional

import numpy as np
import soundfile as sf
import torch

logger = logging.getLogger("voice_studio")

DEFAULT_MODEL_ID = "k2-fsa/OmniVoice"

# Preset of the reference studio ("Standard quality preset").
DEFAULTS = {
    "mode": "Voice Clone",
    "language": "Auto",
    "speed": 1.0,
    "num_step": 32,
    "guidance_scale": 2.0,
    "denoise": True,
    "preprocess_prompt": True,
    "postprocess_output": True,
    "normalize_text": False,
    "pad_duration": 0.1,
    "fade_duration": 0.1,
    "seed": 42,
    "audio_chunk_duration": 15.0,
    "audio_chunk_threshold": 30.0,
    "output_filename": "omnivoice_turbo.wav",
}

# Non-verbal tags supported by OmniVoice (README, "Non-Verbal & Pronunciation").
EXPRESSION_TAGS = [
    "[laughter]",
    "[sigh]",
    "[confirmation-en]",
    "[question-en]",
    "[question-ah]",
    "[question-oh]",
    "[question-ei]",
    "[question-yi]",
    "[surprise-ah]",
    "[surprise-oh]",
    "[surprise-wa]",
    "[surprise-yo]",
    "[dissatisfaction-hnn]",
]

# Voice Design attributes, as offered by the official omnivoice-demo.
VOICE_DESIGN_CATEGORIES = {
    "Gender": ["Male", "Female"],
    "Age": ["Child", "Teenager", "Young Adult", "Middle-aged", "Elderly"],
    "Pitch": [
        "Very Low Pitch",
        "Low Pitch",
        "Moderate Pitch",
        "High Pitch",
        "Very High Pitch",
    ],
    "Style": ["Whisper"],
    "English Accent": [
        "American Accent",
        "Australian Accent",
        "British Accent",
        "Chinese Accent",
        "Canadian Accent",
        "Indian Accent",
        "Korean Accent",
        "Portuguese Accent",
        "Russian Accent",
        "Japanese Accent",
    ],
    "Chinese Dialect": [
        "Henan Dialect / 河南话",
        "Shaanxi Dialect / 陕西话",
        "Sichuan Dialect / 四川话",
        "Guizhou Dialect / 贵州话",
        "Yunnan Dialect / 云南话",
        "Guilin Dialect / 桂林话",
        "Jinan Dialect / 济南话",
        "Shijiazhuang Dialect / 石家庄话",
        "Gansu Dialect / 甘肃话",
        "Ningxia Dialect / 宁夏话",
        "Qingdao Dialect / 青岛话",
        "Northeast Dialect / 东北话",
    ],
}


class StudioError(Exception):
    """User-facing error (bad input, missing reference, ...)."""


def build_instruct(selected, extra: Optional[str] = None) -> Optional[str]:
    """Turn Voice Design dropdown values into an OmniVoice ``instruct`` string.

    Dialects are passed in Chinese (the model has no English name for them);
    everything else in English. Validation and language unification are done
    by OmniVoice itself (``_resolve_instruct``).
    """
    parts = []
    for value in selected:
        if not value or value == "Auto":
            continue
        if " / " in value:
            parts.append(value.split(" / ", 1)[1].strip())
        else:
            parts.append(value.strip().lower())
    if extra and extra.strip():
        parts.extend(p.strip() for p in re.split(r"[,，]", extra) if p.strip())
    return ", ".join(parts) if parts else None


def language_choices():
    """``["Auto", <600+ display names>]`` exactly like the official demo."""
    from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name

    return ["Auto"] + sorted(lang_display_name(n) for n in LANG_NAMES)


def _language_to_engine(display: Optional[str]) -> Optional[str]:
    if not display or display == "Auto":
        return None
    from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name

    for name in LANG_NAMES:
        if lang_display_name(name) == display:
            return name
    return display


def _sha1_file(path: str) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def _safe_stem(filename: Optional[str]) -> str:
    stem = os.path.basename((filename or "").strip())
    stem = re.sub(r"\.(wav|wave)$", "", stem, flags=re.IGNORECASE)
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("._")
    return stem or os.path.splitext(DEFAULTS["output_filename"])[0]


def _fmt_gb(num_bytes: float) -> str:
    return f"{num_bytes / 1024**3:.2f} GB"


@dataclass
class CachedVoice:
    key: str
    prompt: object  # omnivoice.VoiceClonePrompt
    ref_text: str
    encode_seconds: float
    source: str  # "encoded" | "drive"


@dataclass
class GenerationResult:
    path: str
    filename: str
    sample_rate: int
    audio_seconds: float
    generation_seconds: float
    internal_calls: int
    voice_prompt: str
    peak_vram: Optional[float]
    current_vram: Optional[float]
    notes: list = field(default_factory=list)


class StudioEngine:
    def __init__(
        self,
        model_id: str = DEFAULT_MODEL_ID,
        data_dir: str = "./voice_studio_data",
        device: Optional[str] = None,
        flashinfer: str = "auto",
        cuda_graphs: bool = False,
    ):
        self.model_id = model_id
        self.data_dir = os.path.abspath(data_dir)
        self.outputs_dir = os.path.join(self.data_dir, "outputs")
        self.voices_dir = os.path.join(self.data_dir, "voices")
        self.settings_path = os.path.join(self.data_dir, "settings.json")
        for d in (self.outputs_dir, self.voices_dir):
            os.makedirs(d, exist_ok=True)

        if device is None:
            from omnivoice.utils.common import get_best_device

            device = get_best_device()
        self.device = device
        self.device_type = torch.device(device).type
        self.dtype = torch.float16 if self.device_type in ("cuda", "mps", "xpu") else torch.float32
        self.flashinfer_mode = flashinfer
        self.cuda_graphs = cuda_graphs

        self.model = None
        self.sampling_rate = 24000
        self.flashinfer_active = False
        self.flashinfer_note = ""
        self.load_seconds = 0.0

        self._lock = threading.RLock()
        self._voices: dict = {}
        self._calls = 0
        self._steps = 0
        self._progress_hook: Optional[Callable[[int, int], None]] = None

    # ------------------------------------------------------------------
    # Loading
    # ------------------------------------------------------------------

    def _load_model(self):
        from omnivoice import OmniVoice

        device_map = "cuda:0" if self.device == "cuda" else self.device
        self.model = OmniVoice.from_pretrained(
            self.model_id, device_map=device_map, dtype=self.dtype
        )
        self.model.eval()
        self.sampling_rate = int(self.model.sampling_rate)

    def load(self):
        t0 = time.perf_counter()
        self._load_model()
        self.flashinfer_active, self.flashinfer_note = self._maybe_enable_flashinfer()
        self._install_instrumentation()
        self.load_seconds = time.perf_counter() - t0
        logger.info("Model ready in %.1fs (%s)", self.load_seconds, self.acceleration_label)
        return self

    def _smoke_test(self):
        audio = self.model.generate(text="Hello, this is a quick self test.", num_step=8)[0]
        if audio.size == 0 or not np.isfinite(audio).all():
            raise RuntimeError("self-test produced empty or non-finite audio")

    def _maybe_enable_flashinfer(self):
        mode = (self.flashinfer_mode or "auto").lower()
        if mode == "off":
            return False, "disabled"
        if self.device_type != "cuda":
            return False, "requires an NVIDIA GPU"
        if self.dtype != torch.float16:
            return False, "requires FP16"
        cc = torch.cuda.get_device_capability(0)
        if mode == "auto" and cc < (8, 0):
            return False, f"skipped on sm_{cc[0]}{cc[1]} (auto mode enables it on sm_80+)"
        try:
            import flashinfer  # noqa: F401
            from omnivoice.models.omnivoice_flashinfer import apply_flashinfer
        except Exception as e:  # not installed / incompatible build
            return False, f"unavailable ({type(e).__name__})"
        try:
            apply_flashinfer(self.model, enable_cuda_graph=self.cuda_graphs)
            self._smoke_test()
            return True, "self-test passed"
        except Exception as e:
            logger.warning("FlashInfer failed (%s); reloading the baseline model.", e)
            self.model = None
            gc.collect()
            torch.cuda.empty_cache()
            self._load_model()
            return False, f"self-test failed, fell back to baseline ({type(e).__name__})"

    def adopt(self, other: "StudioEngine") -> "StudioEngine":
        """Take over the model already loaded by ``other`` (notebook re-runs
        that re-import this module) instead of loading the weights again."""
        self.model = other.model
        self.sampling_rate = other.sampling_rate
        self.flashinfer_active = other.flashinfer_active
        self.flashinfer_note = other.flashinfer_note
        self.load_seconds = other.load_seconds
        self.cuda_graphs = other.cuda_graphs
        if other.model_id == self.model_id:
            self._voices.update(other._voices)
        self._install_instrumentation()
        return self

    def _install_instrumentation(self):
        """Count calls of OmniVoice's own iterative decoder (and its forward
        steps on the baseline path) for progress and statistics only.
        Idempotent: re-installing replaces the previous wrapper and hook."""
        model = self.model
        inner = getattr(model, "_vs_inner_generate_iterative", None) or model._generate_iterative
        model._vs_inner_generate_iterative = inner
        engine = self

        def counted_generate_iterative(task, gen_config):
            engine._calls += 1
            engine._steps = 0
            engine._report()
            return inner(task, gen_config)

        model._generate_iterative = counted_generate_iterative

        def forward_hook(module, args, output):
            engine._steps += 1
            engine._report()

        previous = getattr(model, "_vs_hook_handle", None)
        if previous is not None:
            previous.remove()
        model._vs_hook_handle = model.register_forward_hook(forward_hook)

    def _report(self):
        if self._progress_hook is not None:
            try:
                self._progress_hook(self._calls, self._steps)
            except Exception:  # progress must never break generation
                pass

    # ------------------------------------------------------------------
    # Labels / environment
    # ------------------------------------------------------------------

    @property
    def gpu_name(self) -> str:
        if self.device_type == "cuda":
            return torch.cuda.get_device_name(0)
        return self.device_type.upper()

    @property
    def gpu_total_gb(self) -> Optional[float]:
        if self.device_type == "cuda":
            return torch.cuda.mem_get_info(0)[1] / 1024**3
        return None

    @property
    def acceleration_label(self) -> str:
        precision = "FP16" if self.dtype == torch.float16 else "FP32"
        if self.flashinfer_active:
            label = f"OmniVoice {precision} + FlashInfer"
            if self.cuda_graphs:
                label += " + CUDA graphs"
            return label
        if self.device_type == "cuda" and "T4" in self.gpu_name:
            return f"OmniVoice {precision} — T4-safe baseline"
        return f"OmniVoice {precision} — baseline"

    @property
    def badge_text(self) -> str:
        parts = [self.gpu_name.replace("NVIDIA ", "")]
        if self.gpu_total_gb is not None:
            parts.append(f"{self.gpu_total_gb:.1f} GB")
        parts.append(self.acceleration_label)
        return " · ".join(parts)

    def vram(self):
        if self.device_type != "cuda":
            return None, None
        return torch.cuda.max_memory_allocated(0), torch.cuda.memory_allocated(0)

    # ------------------------------------------------------------------
    # Settings persistence
    # ------------------------------------------------------------------

    def load_settings(self) -> dict:
        settings = dict(DEFAULTS)
        try:
            with open(self.settings_path, encoding="utf-8") as f:
                saved = json.load(f)
            settings.update({k: v for k, v in saved.items() if k in DEFAULTS})
        except (OSError, ValueError):
            pass
        return settings

    def save_settings(self, settings: dict):
        data = {k: settings[k] for k in DEFAULTS if k in settings}
        tmp = self.settings_path + ".tmp"
        try:
            with open(tmp, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            os.replace(tmp, self.settings_path)
        except OSError as e:
            logger.warning("Could not save settings: %s", e)

    # ------------------------------------------------------------------
    # Voice prompt cache
    # ------------------------------------------------------------------

    def voice_key(self, ref_audio: str, ref_text: Optional[str], preprocess: bool) -> str:
        transcript = (ref_text or "").strip() or "<auto-transcribe>"
        raw = f"{_sha1_file(ref_audio)}|{transcript}|{int(bool(preprocess))}|{self.model_id}"
        return hashlib.sha1(raw.encode("utf-8")).hexdigest()[:20]

    def lookup_voice(self, key: str) -> Optional[CachedVoice]:
        if key in self._voices:
            return self._voices[key]
        pt = os.path.join(self.voices_dir, f"{key}.pt")
        if not os.path.isfile(pt):
            return None
        from omnivoice import VoiceClonePrompt

        try:
            t0 = time.perf_counter()
            prompt = VoiceClonePrompt.load(pt, map_location="cpu")
            meta = {}
            meta_path = os.path.join(self.voices_dir, f"{key}.json")
            if os.path.isfile(meta_path):
                with open(meta_path, encoding="utf-8") as f:
                    meta = json.load(f)
        except Exception as e:
            logger.warning("Ignoring unreadable cached voice %s: %s", pt, e)
            return None
        voice = CachedVoice(
            key=key,
            prompt=prompt,
            ref_text=prompt.ref_text,
            encode_seconds=float(meta.get("encode_seconds", time.perf_counter() - t0)),
            source="drive",
        )
        self._voices[key] = voice
        return voice

    def is_voice_cached(self, ref_audio: Optional[str], ref_text: Optional[str], preprocess: bool) -> bool:
        if not ref_audio or not os.path.isfile(ref_audio):
            return False
        key = self.voice_key(ref_audio, ref_text, preprocess)
        return key in self._voices or os.path.isfile(os.path.join(self.voices_dir, f"{key}.pt"))

    def prepare_voice(self, ref_audio: Optional[str], ref_text: Optional[str], preprocess: bool = True):
        """Return ``(CachedVoice, hit: bool)``; encodes and persists on a miss."""
        if not ref_audio or not os.path.isfile(ref_audio):
            raise StudioError("Upload a reference voice first (3–10 seconds recommended).")
        with self._lock:
            key = self.voice_key(ref_audio, ref_text, preprocess)
            voice = self.lookup_voice(key)
            if voice is not None:
                return voice, True
            transcript = (ref_text or "").strip() or None
            t0 = time.perf_counter()
            try:
                prompt = self.model.create_voice_clone_prompt(
                    ref_audio=ref_audio, ref_text=transcript, preprocess_prompt=bool(preprocess)
                )
            except ValueError as e:
                raise StudioError(str(e)) from e
            if self.device_type == "cuda":
                torch.cuda.synchronize()
            seconds = time.perf_counter() - t0
            voice = CachedVoice(key, prompt, prompt.ref_text, seconds, "encoded")
            self._voices[key] = voice
            try:
                prompt.save(os.path.join(self.voices_dir, f"{key}.pt"))
                with open(os.path.join(self.voices_dir, f"{key}.json"), "w", encoding="utf-8") as f:
                    json.dump(
                        {
                            "ref_text": prompt.ref_text,
                            "user_transcript": transcript,
                            "encode_seconds": seconds,
                            "source_file": os.path.basename(ref_audio),
                            "preprocess_prompt": bool(preprocess),
                            "model_id": self.model_id,
                            "created": time.strftime("%Y-%m-%d %H:%M:%S"),
                        },
                        f,
                        indent=2,
                        ensure_ascii=False,
                    )
            except OSError as e:
                logger.warning("Voice cached in memory only (disk write failed: %s)", e)
            return voice, False

    def clear_voice_cache(self) -> int:
        with self._lock:
            keys = set(self._voices)
            self._voices.clear()
            for name in os.listdir(self.voices_dir):
                if name.endswith((".pt", ".json")):
                    try:
                        os.remove(os.path.join(self.voices_dir, name))
                        keys.add(os.path.splitext(name)[0])
                    except OSError:
                        pass
            return len(keys)

    # ------------------------------------------------------------------
    # GPU / graph caches
    # ------------------------------------------------------------------

    def clear_gpu_caches(self) -> str:
        with self._lock:
            graphs = 0
            cache = getattr(self.model, "_fi_graph_cache", None)
            if cache:
                graphs = len(cache)
                cache.clear()
            gc.collect()
            if self.device_type == "cuda":
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
                torch.cuda.reset_peak_memory_stats(0)
                reserved = torch.cuda.memory_reserved(0)
                allocated = torch.cuda.memory_allocated(0)
                return (
                    f"Released cached GPU memory. Captured CUDA graphs cleared: {graphs}. "
                    f"Allocated: {_fmt_gb(allocated)} · Reserved: {_fmt_gb(reserved)}"
                )
            return f"Python garbage collected (device: {self.device_type}, no GPU memory to release)."

    # ------------------------------------------------------------------
    # Generation
    # ------------------------------------------------------------------

    def estimate_internal_calls(self, kw: dict, gen_config) -> Optional[int]:
        """How many times OmniVoice's iterative decoder will run, mirroring
        ``OmniVoice.generate`` / ``_generate_chunked`` (read-only)."""
        try:
            from omnivoice.utils.text import chunk_text_punctuation

            task = self.model._preprocess_all(
                text=kw["text"],
                language=kw.get("language"),
                voice_clone_prompt=kw.get("voice_clone_prompt"),
                instruct=kw.get("instruct"),
                preprocess_prompt=gen_config.preprocess_prompt,
                speed=kw.get("speed"),
                normalize_text=kw.get("normalize_text", False),
            )
            frame_rate = self.model.audio_tokenizer.config.frame_rate
            target = task.target_lens[0]
            if target <= int(gen_config.audio_chunk_threshold * frame_rate):
                return 1
            text = task.texts[0]
            chunk_len = int(gen_config.audio_chunk_duration * frame_rate / (target / len(text)))
            return max(1, len(chunk_text_punctuation(text=text, chunk_len=chunk_len, min_chunk_len=3)))
        except Exception:
            return None

    def generate(
        self,
        *,
        mode: str,
        text: str,
        language: str = "Auto",
        speed: float = 1.0,
        ref_audio: Optional[str] = None,
        ref_text: Optional[str] = None,
        instruct: Optional[str] = None,
        num_step: int = 32,
        guidance_scale: float = 2.0,
        denoise: bool = True,
        preprocess_prompt: bool = True,
        postprocess_output: bool = True,
        normalize_text: bool = False,
        pad_duration: float = 0.1,
        fade_duration: float = 0.1,
        audio_chunk_duration: float = 15.0,
        audio_chunk_threshold: float = 30.0,
        seed: Optional[int] = 42,
        output_filename: str = DEFAULTS["output_filename"],
        progress: Optional[Callable[[float, str], None]] = None,
    ) -> GenerationResult:
        from omnivoice import OmniVoiceGenerationConfig
        from omnivoice.utils.common import fix_random_seed

        text = (text or "").strip()
        if not text:
            raise StudioError("Please enter the text to synthesize.")
        if self.model is None:
            raise StudioError("The model is not loaded yet.")

        gen_config = OmniVoiceGenerationConfig(
            num_step=int(num_step),
            guidance_scale=float(guidance_scale),
            denoise=bool(denoise),
            preprocess_prompt=bool(preprocess_prompt),
            postprocess_output=bool(postprocess_output),
            audio_chunk_duration=float(audio_chunk_duration),
            audio_chunk_threshold=float(audio_chunk_threshold),
            pad_duration=float(pad_duration),
            fade_duration=float(fade_duration),
        )
        notes = []
        kw = dict(
            text=text,
            language=_language_to_engine(language),
            speed=float(speed),
            generation_config=gen_config,
        )

        with self._lock:
            voice_prompt = "not used (" + mode + ")"
            if mode == "Voice Clone":
                if progress:
                    progress(0.0, "Preparing reference voice…")
                voice, hit = self.prepare_voice(ref_audio, ref_text, preprocess_prompt)
                kw["voice_clone_prompt"] = voice.prompt
                if hit:
                    voice_prompt = "cache hit ✅"
                else:
                    voice_prompt = f"encoded now in {voice.encode_seconds:.2f}s, cached for next runs"
            elif mode == "Voice Design":
                if not instruct:
                    raise StudioError("Voice Design needs at least one voice attribute (gender, age, pitch, style, accent or dialect).")
                kw["instruct"] = instruct

            if normalize_text:
                try:
                    from omnivoice.utils.text import normalize_text as _nt

                    _nt("1", kw["language"])
                    kw["normalize_text"] = True
                except ImportError:
                    notes.append("Normalize text skipped: install `omnivoice[tn]` (WeTextProcessing).")

            total_calls = self.estimate_internal_calls(kw, gen_config) or 1
            steps = max(1, gen_config.num_step)

            def hook(calls, step):
                if progress:
                    done = (max(calls - 1, 0) + min(step, steps) / steps) / total_calls
                    progress(min(done, 0.99), f"OmniVoice decoder call {calls}/{max(total_calls, calls)}")

            if seed is not None and str(seed).strip() not in ("", "-1"):
                fix_random_seed(int(seed))
            if self.device_type == "cuda":
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats(0)

            self._calls, self._steps = 0, 0
            self._progress_hook = hook
            t0 = time.perf_counter()
            try:
                audio = self.model.generate(**kw)[0]
            except ValueError as e:  # e.g. invalid voice-design attributes
                raise StudioError(str(e)) from e
            finally:
                self._progress_hook = None
            if self.device_type == "cuda":
                torch.cuda.synchronize()
            gen_seconds = time.perf_counter() - t0
            calls = self._calls
            peak, current = self.vram()

        audio = np.clip(np.asarray(audio, dtype=np.float32), -1.0, 1.0)
        filename = f"{_safe_stem(output_filename)}_{uuid.uuid4().hex[:7]}.wav"
        path = os.path.join(self.outputs_dir, filename)
        sf.write(path, audio, self.sampling_rate, subtype="PCM_16")
        return GenerationResult(
            path=path,
            filename=filename,
            sample_rate=self.sampling_rate,
            audio_seconds=audio.shape[-1] / self.sampling_rate,
            generation_seconds=gen_seconds,
            internal_calls=calls,
            voice_prompt=voice_prompt,
            peak_vram=peak,
            current_vram=current,
            notes=notes,
        )

In [ ]:
%%writefile {APP_DIR}/voice_studio/ui.py
"""Gradio interface of VOICE STUDIO DA GALAXIA.

Layout, labels, defaults and ranges follow the reference "OmniVoice Turbo
Studio" screenshots; the only identity change is the title.
"""

import html
import inspect
import logging
import os
import traceback

import gradio as gr

from .engine import (
    DEFAULTS,
    EXPRESSION_TAGS,
    VOICE_DESIGN_CATEGORIES,
    StudioEngine,
    StudioError,
    build_instruct,
    language_choices,
)

logger = logging.getLogger("voice_studio")

TITLE = "🌌 VOICE STUDIO DA GALAXIA"
PAGE_TITLE = "VOICE STUDIO DA GALAXIA"
SUBTITLE = "600+ languages · voice cloning · native long-form narration"
MODES = ["Voice Clone", "Voice Design", "Auto Voice"]
_INSERTED = "1"

CSS_PATH = os.path.join(os.path.dirname(__file__), "theme.css")

# Forces the dark palette of the reference, whatever the OS/browser setting.
FORCE_DARK_JS = """
() => {
  document.documentElement.classList.add('dark');
  document.body.classList.add('dark');
  document.querySelectorAll('gradio-app, .gradio-container').forEach(el => el.classList.add('dark'));
}
"""

# Inserts the selected expression tag at the caret of the text box.
INSERT_TAG_JS = """
(text, tag, flag) => {
  if (!tag) return [text, tag, ''];
  const s = text ?? '';
  const ta = document.querySelector('#vs-text textarea');
  let start = s.length, end = s.length;
  if (ta && ta.value === s && typeof ta.selectionStart === 'number') {
    start = ta.selectionStart; end = ta.selectionEnd;
  }
  const before = s.slice(0, start), after = s.slice(end);
  const left = before && !/\\s$/.test(before) ? ' ' : '';
  const right = after && !/^\\s/.test(after) ? ' ' : '';
  return [before + left + tag + right + after, tag, '1'];
}
"""

READY_STATUS = "### 🟣 Ready\n\nEnter text, pick a mode and press **Generate Audio**."
TIP_MD = (
    "**For maximum repeat-generation speed:** Prepare/cache the reference once, "
    "then keep the same reference while changing only the target text."
)


def _counter(text):
    text = text or ""
    return f"**{len(text):,} characters · {len(text.split()):,} words**"


def _voice_status(cached, encoded_s=None, source=None):
    if not cached:
        return (
            "### ⚪ Voice not cached\n\n"
            "Press **Prepare / Cache Voice** to encode the reference once "
            "(Generate Audio also caches it automatically)."
        )
    if source == "drive":
        detail = "Reference restored from the Drive cache. Future generations with this same reference skip that work."
    elif encoded_s is not None:
        detail = f"Reference encoded in **{encoded_s:.2f}s**. Future generations with this same reference skip that work."
    else:
        detail = "Future generations with this same reference skip that work."
    return f"### 🟢 Voice cached\n\n{detail}"


def _finished_md(engine, res, steps, cfg):
    preset = "Standard quality preset" if int(steps) == 32 and float(cfg) == 2.0 else "Custom quality preset"
    rtf = res.generation_seconds / res.audio_seconds if res.audio_seconds else 0.0
    speed_x = res.audio_seconds / res.generation_seconds if res.generation_seconds else 0.0
    lines = [
        f"**Acceleration:** {engine.acceleration_label}",
        f"**Quality:** {preset} · **Steps:** {int(steps)} · **CFG:** {float(cfg):.1f}",
        f"**Voice prompt:** {res.voice_prompt}",
        f"**Internal generation calls:** {res.internal_calls}",
        f"**Audio:** {res.audio_seconds:.1f}s · **Generation:** {res.generation_seconds:.2f}s",
        f"**RTF:** {rtf:.3f} · **Throughput:** {speed_x:.1f}× realtime",
    ]
    if res.peak_vram is not None:
        lines.append(
            f"**Peak VRAM:** {res.peak_vram / 1024**3:.2f} GB · "
            f"**Current VRAM:** {res.current_vram / 1024**3:.2f} GB"
        )
    lines.append(f"**File:** `{res.filename}`")
    lines.extend(f"⚠️ {n}" for n in res.notes)
    return "### ✅ Finished\n\n" + "  \n".join(lines)


def _header_html(engine):
    return (
        '<div id="vs-header">'
        f"<h1>{html.escape(TITLE)}</h1>"
        f'<p class="vs-subtitle">{html.escape(SUBTITLE)}</p>'
        f'<span class="vs-badge">{html.escape(engine.badge_text)}</span>'
        "</div>"
    )


def _footer_md(engine):
    return (
        f"**Engine:** official latest `k2-fsa/OmniVoice` · "
        f"**Acceleration:** `{engine.acceleration_label}` "
        f"**GPU:** `{engine.gpu_name.replace('NVIDIA ', '')}` · "
        f"**Sample rate:** `{engine.sampling_rate:,} Hz`\n\n"
        "Progress shown above is tied to calls of OmniVoice's own iterative decoder. "
        "It does **not** replace or alter OmniVoice's native long-form generation algorithm."
    )


def build_demo(engine: StudioEngine, restore_settings: bool = True, **blocks_kwargs) -> gr.Blocks:
    s = engine.load_settings() if restore_settings else dict(DEFAULTS)
    langs = language_choices()
    if s["language"] not in langs:
        s["language"] = "Auto"
    if s["mode"] not in MODES:
        s["mode"] = DEFAULTS["mode"]

    with gr.Blocks(title=PAGE_TITLE, analytics_enabled=False, **blocks_kwargs) as demo:
        gr.HTML(_header_html(engine))

        with gr.Row(equal_height=False, elem_id="vs-main"):
            # ============================ LEFT ============================
            with gr.Column(scale=7, min_width=420):
                with gr.Group(elem_classes="vs-panel"):
                    mode = gr.Radio(MODES, value=s["mode"], label="Mode", elem_id="vs-mode")
                    text = gr.Textbox(
                        label="Text to synthesize",
                        lines=12,
                        max_lines=40,
                        placeholder=(
                            "Paste short speech or a long narration here. OmniVoice will "
                            "automatically use its native long-form chunking."
                        ),
                        elem_id="vs-text",
                    )
                counter = gr.Markdown(_counter(""), elem_id="vs-counter")

                with gr.Group(elem_classes="vs-panel"):
                    with gr.Row():
                        language = gr.Dropdown(langs, value=s["language"], label="Language", filterable=True)
                        speed = gr.Slider(
                            0.5, 1.5, value=s["speed"], step=0.05,
                            label="Speech speed",
                            info=">1 faster speech · <1 slower speech",
                        )

                with gr.Row(equal_height=False):
                    with gr.Column(scale=5, min_width=200):
                        tag = gr.Dropdown(EXPRESSION_TAGS, value=None, label="Optional expression tag", elem_id="vs-tag")
                    with gr.Column(scale=2, min_width=120):
                        insert_btn = gr.Button("Insert Tag", variant="secondary", elem_id="vs-insert")
                insert_flag = gr.Textbox(visible=False)

                # ---------------- Voice Clone ----------------
                with gr.Group(visible=s["mode"] == "Voice Clone", elem_classes="vs-panel vs-section") as clone_group:
                    gr.Markdown("### 🎤 Voice Clone", elem_classes="vs-section-title")
                    ref_audio = gr.Audio(
                        label="Reference voice (3–10 seconds recommended)",
                        type="filepath",
                        sources=["upload", "microphone"],
                        elem_id="vs-ref-audio",
                    )
                    ref_text = gr.Textbox(
                        label="Exact reference transcript",
                        lines=3,
                        elem_id="vs-ref-text",
                        placeholder=(
                            "Type exactly what is spoken in the reference audio. "
                            "Leave empty to auto-transcribe with Whisper."
                        ),
                    )
                    with gr.Row():
                        prepare_btn = gr.Button("🚀 Prepare / Cache Voice", variant="secondary")
                        clear_voice_btn = gr.Button("Clear Voice Cache", variant="secondary")
                    voice_status = gr.Markdown(_voice_status(False), elem_id="vs-voice-status")

                # ---------------- Voice Design ----------------
                with gr.Group(visible=s["mode"] == "Voice Design", elem_classes="vs-panel vs-section") as design_group:
                    gr.Markdown("### 🎨 Voice Design", elem_classes="vs-section-title")
                    design_dropdowns = []
                    cats = list(VOICE_DESIGN_CATEGORIES.items())
                    for row_start in range(0, len(cats), 3):
                        with gr.Row():
                            for cat, choices in cats[row_start:row_start + 3]:
                                info = {
                                    "English Accent": "Only effective for English speech.",
                                    "Chinese Dialect": "Only effective for Chinese speech.",
                                }.get(cat)
                                design_dropdowns.append(
                                    gr.Dropdown(
                                        ["Auto"] + choices, value="Auto", label=cat, info=info,
                                        elem_id="vs-design-" + cat.lower().replace(" ", "-"),
                                    )
                                )
                    design_extra = gr.Textbox(
                        label="Voice attributes (optional, comma-separated)",
                        placeholder="e.g. female, low pitch, british accent",
                    )

            # ============================ RIGHT ===========================
            with gr.Column(scale=5, min_width=320):
                gr.Markdown("### 🔊 Output", elem_id="vs-output-title")
                audio_out = gr.Audio(
                    label="Generated audio",
                    type="filepath",
                    interactive=False,
                    elem_id="vs-audio-out",
                )
                file_out = gr.File(label="Download WAV", interactive=False, elem_id="vs-file-out")
                status = gr.Markdown(READY_STATUS, elem_id="vs-status")
                gr.Markdown(TIP_MD, elem_id="vs-tip")

        # ======================= FULL WIDTH ===============================
        with gr.Accordion("📚 Long-form Narration", open=True):
            gr.Markdown(
                "These are OmniVoice's **native long-form controls**. The model itself "
                "performs the text segmentation and audio joining."
            )
            with gr.Row():
                chunk_dur = gr.Slider(
                    8, 25, value=s["audio_chunk_duration"], step=1,
                    label="Target internal chunk duration (seconds)",
                    info="15s is the official default.",
                )
                chunk_thr = gr.Slider(
                    15, 60, value=s["audio_chunk_threshold"], step=1,
                    label="Activate long-form chunking above (seconds)",
                    info="30s is the official default.",
                )

        with gr.Accordion("⚙️ Quality / Advanced Settings", open=False):
            gr.Markdown(
                "**Keep 32 steps and CFG 2.0 if you do not want to trade quality for speed.** "
                "FlashInfer, prompt caching and the newer inference path provide the speedup "
                "without lowering these."
            )
            with gr.Row():
                num_step = gr.Slider(
                    4, 64, value=s["num_step"], step=1, label="Inference steps",
                    info="32 = standard. Lower values can be faster but may change quality.",
                    elem_id="vs-steps",
                )
                cfg = gr.Slider(0.0, 4.0, value=s["guidance_scale"], step=0.1, label="Guidance scale (CFG)")
            with gr.Row():
                denoise = gr.Checkbox(value=s["denoise"], label="Denoise")
                preprocess = gr.Checkbox(value=s["preprocess_prompt"], label="Preprocess reference voice")
                postprocess = gr.Checkbox(value=s["postprocess_output"], label="Postprocess generated audio")
                normalize = gr.Checkbox(
                    value=s["normalize_text"], label="Normalize text",
                    info="Useful for numbers/dates when supported.",
                )
            with gr.Row():
                pad = gr.Slider(0.0, 0.5, value=s["pad_duration"], step=0.01, label="Edge padding (seconds)")
                fade = gr.Slider(0.0, 0.5, value=s["fade_duration"], step=0.01, label="Fade duration (seconds)")
                seed = gr.Number(value=s["seed"], precision=0, label="Seed")
            out_name = gr.Textbox(value=s["output_filename"], label="Output filename", max_lines=1)
            clear_gpu_btn = gr.Button("🧹 Clear GPU / Graph Caches", variant="secondary")

        with gr.Row(equal_height=False):
            generate_btn = gr.Button("✨ Generate Audio", variant="primary", size="lg", scale=7, elem_id="vs-generate")
            clear_btn = gr.Button("Clear Text / Output", variant="secondary", size="lg", scale=2, elem_id="vs-clear")

        gr.Markdown("---")
        gr.Markdown(_footer_md(engine), elem_id="vs-footer")

        # ======================= CALLBACKS ================================
        text.change(_counter, text, counter, queue=False, show_progress="hidden")

        def on_mode(m):
            return gr.update(visible=m == "Voice Clone"), gr.update(visible=m == "Voice Design")

        mode.change(on_mode, mode, [clone_group, design_group], queue=False, show_progress="hidden")

        def on_insert(t, chosen, flag):
            if not chosen:
                gr.Info("Choose an expression tag first.")
                return t
            if flag == _INSERTED:
                return t
            t = t or ""
            return (t + (" " if t and not t.endswith((" ", "\n")) else "") + chosen)

        insert_btn.click(
            on_insert, [text, tag, insert_flag], text, js=INSERT_TAG_JS, queue=False, show_progress="hidden"
        )

        def refresh_voice(audio_path, transcript, pre):
            try:
                return _voice_status(engine.is_voice_cached(audio_path, transcript, pre))
            except OSError:
                return _voice_status(False)

        for comp in (ref_audio, preprocess):
            comp.change(refresh_voice, [ref_audio, ref_text, preprocess], voice_status, show_progress="hidden")
        ref_text.blur(refresh_voice, [ref_audio, ref_text, preprocess], voice_status, show_progress="hidden")

        def on_prepare(audio_path, transcript, pre):
            try:
                voice, hit = engine.prepare_voice(audio_path, transcript, pre)
            except StudioError as e:
                gr.Warning(str(e))
                return _voice_status(False), gr.update()
            except Exception as e:
                logger.error("Prepare failed:\n%s", traceback.format_exc())
                return f"❌ **Voice preparation failed**\n\n`{type(e).__name__}: {e}`", gr.update()
            md = _voice_status(True, voice.encode_seconds, voice.source)
            new_text = gr.update(value=voice.ref_text) if not (transcript or "").strip() else gr.update()
            return md, new_text

        prepare_btn.click(on_prepare, [ref_audio, ref_text, preprocess], [voice_status, ref_text])

        def on_clear_voice():
            n = engine.clear_voice_cache()
            gr.Info(f"Voice cache cleared ({n} item(s)).")
            return _voice_status(False)

        clear_voice_btn.click(on_clear_voice, None, voice_status)

        def on_clear_gpu():
            msg = engine.clear_gpu_caches()
            gr.Info("GPU / graph caches cleared.")
            return f"### 🧹 Caches cleared\n\n{msg}"

        clear_gpu_btn.click(on_clear_gpu, None, status)

        def on_generate(
            m, t, lang, spd, r_audio, r_text, *rest, progress=gr.Progress()
        ):
            n = len(design_dropdowns)
            design_values, rest = rest[:n], rest[n:]
            (d_extra, c_dur, c_thr, steps, g_cfg, dn, pre, post, norm, p_pad, p_fade, sd, name) = rest
            settings = dict(
                mode=m, language=lang, speed=spd, num_step=steps, guidance_scale=g_cfg,
                denoise=dn, preprocess_prompt=pre, postprocess_output=post, normalize_text=norm,
                pad_duration=p_pad, fade_duration=p_fade, seed=sd,
                audio_chunk_duration=c_dur, audio_chunk_threshold=c_thr, output_filename=name,
            )
            engine.save_settings(settings)
            no_change = gr.update()
            try:
                res = engine.generate(
                    mode=m, text=t, language=lang, speed=spd,
                    ref_audio=r_audio, ref_text=r_text,
                    instruct=build_instruct(design_values, d_extra),
                    num_step=steps, guidance_scale=g_cfg, denoise=dn,
                    preprocess_prompt=pre, postprocess_output=post, normalize_text=norm,
                    pad_duration=p_pad, fade_duration=p_fade,
                    audio_chunk_duration=c_dur, audio_chunk_threshold=c_thr,
                    seed=sd, output_filename=name,
                    progress=lambda frac, desc: progress(frac, desc=desc),
                )
            except StudioError as e:
                gr.Warning(str(e))
                return no_change, no_change, f"### ⚠️ Check the inputs\n\n{e}", no_change, no_change
            except Exception as e:
                logger.error("Generation failed:\n%s", traceback.format_exc())
                hint = ""
                if "out of memory" in str(e).lower():
                    hint = "\n\nTry **Clear GPU / Graph Caches** or a shorter chunk duration."
                return no_change, no_change, f"### ❌ Generation failed\n\n`{type(e).__name__}: {e}`{hint}", no_change, no_change

            voice_md, transcript_update = no_change, no_change
            if m == "Voice Clone":
                voice_md = _voice_status(True)
                try:
                    voice = engine.lookup_voice(engine.voice_key(r_audio, r_text, pre))
                    voice_md = _voice_status(True, voice.encode_seconds, voice.source)
                    if not (r_text or "").strip():
                        transcript_update = gr.update(value=voice.ref_text)
                except Exception:
                    pass
            return res.path, res.path, _finished_md(engine, res, steps, g_cfg), voice_md, transcript_update

        generate_btn.click(
            on_generate,
            [mode, text, language, speed, ref_audio, ref_text, *design_dropdowns, design_extra,
             chunk_dur, chunk_thr, num_step, cfg, denoise, preprocess, postprocess, normalize,
             pad, fade, seed, out_name],
            [audio_out, file_out, status, voice_status, ref_text],
            concurrency_limit=1,
        )

        clear_btn.click(
            lambda: ("", None, None, READY_STATUS),
            None,
            [text, audio_out, file_out, status],
            queue=False,
        )

    return demo


def make_theme():
    return gr.themes.Soft(primary_hue="indigo", secondary_hue="indigo", neutral_hue="gray")


def launch(
    engine: StudioEngine,
    *,
    share: bool = False,
    server_name: str = "0.0.0.0",
    server_port: int = 7860,
    restore_settings: bool = True,
    prevent_thread_lock: bool = False,
    quiet: bool = False,
):
    with open(CSS_PATH, encoding="utf-8") as f:
        css = f.read()
    style = dict(theme=make_theme(), css=css, js=FORCE_DARK_JS)
    # Gradio 6 takes theme/css/js in launch(); Gradio 5 in the Blocks constructor.
    style_in_launch = "theme" in inspect.signature(gr.Blocks.launch).parameters
    demo = build_demo(engine, restore_settings=restore_settings, **({} if style_in_launch else style))
    demo.queue(default_concurrency_limit=1, max_size=32)
    demo.launch(
        share=share,
        server_name=server_name,
        server_port=server_port,
        allowed_paths=[engine.outputs_dir],
        prevent_thread_lock=prevent_thread_lock,
        quiet=quiet,
        show_error=True,
        inline=False,
        **(style if style_in_launch else {}),
    )
    return demo

In [ ]:
%%writefile {APP_DIR}/voice_studio/theme.css
/* VOICE STUDIO DA GALAXIA — layer on top of gr.themes.Soft (dark).
   Palette measured on the reference screenshots:
   page #0b0f19 · panels #1f2937 · inputs #374151 · purple #4f46e5 · secondary #4b5563 */

/* ---------- header ---------- */
.gradio-container { max-width: 1240px !important; margin-left: auto !important; margin-right: auto !important; }
#vs-header { text-align: center; padding: 10px 0 6px; }
#vs-header h1 {
  font-size: 37px !important;
  font-weight: 700;
  line-height: 1.25;
  margin: 0 0 6px;
  color: #ffffff;
}
#vs-header .vs-subtitle { margin: 0 0 9px; font-size: 16.8px; color: #d1d5db; }
#vs-header .vs-badge {
  display: inline-block;
  padding: 3px 14px;
  border: 1px solid #6b7280;
  border-radius: 999px;
  font-size: 16.8px;
  color: #f3f4f6;
}

/* ---------- left column ---------- */
#vs-text textarea { min-height: 250px; }
#vs-counter { padding: 2px 0; }
#vs-counter p { font-weight: 700; margin: 0; }
#vs-insert { align-self: flex-start; min-height: 40px; font-size: 1.1em; }

.vs-section-title { padding: 2px 0 2px 0 !important; }
.vs-section-title h3 { margin: 0 !important; font-size: 1.2em !important; }

#vs-voice-status h3 { margin: 4px 0 10px !important; font-size: 1.2em !important; }
#vs-voice-status p { margin: 6px 0; }

/* ---------- right column ---------- */
#vs-output-title h3, .vs-section-title h3 { font-weight: 700; }
#vs-status h3 { margin: 6px 0 14px !important; font-size: 1.2em !important; }
#vs-status p, #vs-tip p { line-height: 1.6; }

/* ---------- action row ---------- */
#vs-generate { min-height: 54px; font-size: 1.25em !important; }
#vs-clear { align-self: flex-start; min-height: 40px; font-size: 1.15em !important; }

#vs-footer p { margin: 8px 0 14px; }

/* compact numeric box next to slider labels, as in the reference */
.gradio-container .tab-like-container input[type="number"] { width: 58px !important; }

/* file size in the Download WAV row: muted text, as in the reference */
#vs-file-out a { color: #9ca3af !important; text-decoration: none !important; }

In [ ]:
#@title 🧠 4. Baixar e carregar o OmniVoice
import logging, os, time
logging.basicConfig(level=logging.WARNING)
for mod in [m for m in list(sys.modules) if m == "voice_studio" or m.startswith("voice_studio.")]:
    del sys.modules[mod]  # pick up freshly written sources when re-running
from huggingface_hub import snapshot_download
from voice_studio.engine import StudioEngine

if os.path.isdir(MODEL_ID):
    MODEL_PATH = MODEL_ID
else:
    MODEL_PATH = os.path.join(STUDIO_HOME, "models", MODEL_ID.replace("/", "__"))
    t0 = time.perf_counter()
    try:
        snapshot_download(MODEL_ID, local_dir=MODEL_PATH)  # resumes / verifies; instant when cached
    except Exception as e:
        if not os.path.isfile(os.path.join(MODEL_PATH, "config.json")):
            raise
        print("⚠️ Hub indisponível, usando a cópia do Drive:", type(e).__name__)
    print(f"Pesos prontos em {MODEL_PATH} ({time.perf_counter() - t0:.1f}s)")

prev = globals().get("ENGINE")
if prev is not None and getattr(prev, "model", None) is not None and prev.model_id == MODEL_PATH:
    # the module was re-imported above: hand the loaded model to a fresh engine object
    ENGINE = StudioEngine(model_id=MODEL_PATH, data_dir=STUDIO_HOME,
                          flashinfer=FLASHINFER, cuda_graphs=FLASHINFER_CUDA_GRAPHS).adopt(prev)
    print("Modelo já carregado nesta sessão ✔ (para mudar FlashInfer, reinicie o ambiente)")
else:
    if prev is not None:
        prev.model = None
        import gc; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    ENGINE = StudioEngine(model_id=MODEL_PATH, data_dir=STUDIO_HOME,
                          flashinfer=FLASHINFER, cuda_graphs=FLASHINFER_CUDA_GRAPHS).load()
    print(f"Modelo carregado em {ENGINE.load_seconds:.1f}s")
print("Aceleração:", ENGINE.acceleration_label, "| FlashInfer:", ENGINE.flashinfer_note)
print("Badge:", ENGINE.badge_text, "| Sample rate:", ENGINE.sampling_rate)

In [ ]:
#@title 🧩 5. Preparar componentes (aquecimento, cache de vozes, configurações)
import json, time
import numpy as np
from omnivoice.utils.common import fix_random_seed

# Warm-up: initialises CUDA kernels so the first real generation reports honest timings.
fix_random_seed(42)
t0 = time.perf_counter()
_wav = ENGINE.model.generate(text="Warm up.", num_step=8)[0]
if torch.cuda.is_available():
    torch.cuda.synchronize()
assert _wav.size > 0 and np.isfinite(_wav).all(), "warm-up falhou"
print(f"Aquecimento OK ({time.perf_counter() - t0:.2f}s)")

voices = sorted(f for f in os.listdir(ENGINE.voices_dir) if f.endswith(".json"))
print(f"Vozes preparadas no Drive: {len(voices)}")
for f in voices[:20]:
    with open(os.path.join(ENGINE.voices_dir, f), encoding="utf-8") as fh:
        meta = json.load(fh)
    print(f"  • {meta.get('source_file', '?')}: “{meta.get('ref_text', '')[:70]}”")
print("Configurações da interface:", "restauradas do Drive" if RESTORE_LAST_SETTINGS
      and os.path.isfile(ENGINE.settings_path) else "padrão (32 steps · CFG 2.0 · seed 42 · chunk 15s/30s)")

In [ ]:
#@title 🚀 6. Iniciar o Gradio e mostrar o link
from IPython.display import HTML, display
from voice_studio.ui import launch

if globals().get("DEMO") is not None:
    try:
        DEMO.close()
    except Exception:
        pass

port = PORT
for attempt in range(5):
    try:
        DEMO = launch(ENGINE, share=PUBLIC_SHARE_LINK, server_port=port,
                      restore_settings=RESTORE_LAST_SETTINGS, prevent_thread_lock=True, quiet=True)
        break
    except OSError:
        port += 1
else:
    raise RuntimeError("Nenhuma porta livre encontrada")

links = []
if IN_COLAB:
    from google.colab.output import eval_js
    links.append(("Link do Colab (mesma conta Google)", eval_js(f"google.colab.kernel.proxyPort({port})")))
else:
    links.append(("Link local", f"http://127.0.0.1:{port}/"))
if getattr(DEMO, "share_url", None):
    links.append(("Link público gradio.live (72 h)", DEMO.share_url))
for name, url in links:
    print(f"{name}: {url}")
display(HTML("<br>".join(f'<a href="{u}" target="_blank" style="font-size:18px">🌌 {n}</a>' for n, u in links)))
print("O servidor continua rodando em segundo plano. Para reiniciar a interface, execute esta célula de novo.")

---
### Nova sessão / reabrir
Abra o notebook e use **Executar tudo**. A instalação dos pacotes é refeita (o Colab apaga o ambiente),
mas os pesos, as vozes em cache, os WAVs e as configurações vêm do Drive. Se o link parar de responder,
execute a célula **6** novamente.

### Dicas
* **Voice Clone**: referência de 3–10 s + transcrição exata. Deixe a transcrição vazia para usar o Whisper
  (baixado uma vez para o Drive). **Prepare / Cache Voice** codifica a referência uma vez; as gerações
  seguintes com a mesma referência reutilizam o cache (inclusive em sessões futuras).
* **Voice Design**: escolha atributos (gênero, idade, tom, sussurro, sotaque inglês, dialeto chinês).
* **Narração longa**: acima de *Activate long-form chunking above* o próprio OmniVoice divide o texto em
  trechos de ~*Target internal chunk duration* segundos e une o áudio.
* **Clear GPU / Graph Caches** libera memória da GPU (e os CUDA graphs do FlashInfer, se ativos).